https://medium.com/@joshiprerak123/transform-your-audio-denoise-and-enhance-sound-quality-with-python-using-pedalboard-24da7c1df042

In [2]:
import os
import glob

print(os.getcwd())
os.chdir('c:/Users/USER/Documents/코골이/easy_sleep')
print(os.getcwd())
wav_file = glob.glob(os.path.join(os.getcwd(),'*.wav'))
print(wav_file)

c:\Users\USER\Documents\코골이\easy_sleep
c:\Users\USER\Documents\코골이\easy_sleep
['c:\\Users\\USER\\Documents\\코골이\\easy_sleep\\PSG_sample.wav']


In [3]:
from pathlib import Path
wav_files = list(Path('.').glob('*.wav'))
print([str(p) for p in wav_files])

['PSG_sample.wav']


In [4]:
data_path = wav_file[0]
print(data_path)
# 'C:/Users/USER/Documents/코골이/easy_sleep/PSG_sample.wav'

c:\Users\USER\Documents\코골이\easy_sleep\PSG_sample.wav


In [5]:
import librosa
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import noisereduce as nr
import scipy
from scipy import signal
from pathlib import Path
from IPython.display import Audio, display
import soundfile as sf

matplotlib.rcParams['font.family'] = [
    'DejaVu Sans', 'Arial', 'sans-serif', 'Malgun Gothic']

noise reduce 라이브러리를 이용
- stationary 노이즈 제거: 배경 노이즈가 일정한 경우
- non-stationary 노이즈 제거: 변화하는 노이즈일 경우
- nr.reduce_noise: noise 샘플을 직접 제공하는 경우

In [6]:
audio, sr = librosa.load(data_path, sr=16000)

In [7]:
start_minutes = 175
end_minutes = 180
sr = 16000

In [8]:
start_sample = int(start_minutes * 60 * sr)
print(f"start sampel: {start_sample}")
end_sample = int(end_minutes * 60 * sr)  
print(f"end sampel: {end_sample}")

start sampel: 168000000
end sampel: 172800000


In [9]:
print(f"세그먼트 길이: {(end_sample - start_sample)/sr/60:.1f}분") 

세그먼트 길이: 5.0분


In [10]:
# if end_sample > len(audio):
#     end_sample = len(audio)
#     segment = audio[start_sample:end_sample]
# else:
segment = audio[start_sample:end_sample]


In [11]:
display(Audio(segment, rate=sr))

<IPython.lib.display.Audio object>

In [12]:
def stationary_denoise(audio, sr, segment_mins=30):
    """정상 노이즈 처리"""
    segment_length = segment_mins * 60 * sr  # 30분 = 1800초 
    processed_segments = []

    # start, end, stop 
    for i in range(0, len(audio), segment_length): 
        segment = audio[i: i + segment_length]

        # 각 30분 구간마다 stationary 노이즈 제거 
        denoised_segment = nr.reduce_noise(
            y=segment,
            sr=sr, 
            stationary=True,
            prop_decrease=0.9
        )
    
    processed_segments.append(denoised_segment)
    print(f"{segment_mins}분 세그먼트 처리: {i//segment_length +1}")

    return np.concatenate(processed_segments)

In [13]:
stationary_audio = stationary_denoise(segment,sr)
print(f"타입 {type(stationary_audio)}")
print(f"길이 {len(stationary_audio)}")

30분 세그먼트 처리: 1
타입 <class 'numpy.ndarray'>
길이 4800000


In [14]:
print("=== stationary 노이즈 제거 오디오 ===")
Audio(stationary_audio, rate=sr)

=== stationary 노이즈 제거 오디오 ===


<IPython.lib.display.Audio object>

In [ ]:
def non_stationary_denoise(audio, sr, window_mins=5):
    """비정상 노이즈 처리"""
    # size of audio chunk
    window_length = window_mins * 60 * sr 
    processed_segments = []

    # start, end, stop interval(step)
    for i in range(0, len(audio), window_length):
        segment = audio[i: i + window_length]
        
        denoised_segment = nr.reduce_noise(
            y=segment,
            sr=sr,
            stationary=False,
            prop_decrease=0.9
        )
        processed_segments.append(denoised_segment)
    
    return np.concatenate(processed_segments)

    # 과도하면 신호를 왜곡할 수 있음

In [16]:
non_stationary_audio = non_stationary_denoise(segment, sr)
print(f"타입 {type(non_stationary_audio)}")
print(f"길이 {len(non_stationary_audio)}")

타입 <class 'numpy.ndarray'>
길이 4800000


In [17]:
print("=== non-stationary 노이즈 제거 오디오 ===")
Audio(non_stationary_audio, rate=sr)

=== non-stationary 노이즈 제거 오디오 ===


<IPython.lib.display.Audio object>

- 5분 샘플 안: 2분 7초
- 5분 샘플 안: 3분 17초부터 코골이 소리 ~ 3분 33초까지

In [ ]:
def remove_dc_offset(data):
    mean_value = np.mean(data)
    return data - mean_value


def rms_normalize_audio(data, target_rms=0.1):
    rms = np.sqrt(np.mean(data**2))
    # 1e-6을 추가하는 이유는, 0으로 나누는것을 방지하기 위함
    scaling_factor = target_rms / (rms + 1e-6)
    return data * scaling_factor

In [ ]:
def dc_rms_stationary(audio, sr, segment_mins=5):
    audio = remove_dc_offset(audio)
    audio = rms_normalize_audio(audio, target_rms=0.1)
    audio = stationary_denoise(audio, sr, segment_mins=segment_mins)
    return audio


def dc_rms_non_stationary(audio, sr, window_mins=5):
    audio = remove_dc_offset(audio)
    audio = rms_normalize_audio(audio, target_rms=0.1)
    audio = non_stationary_denoise(audio, sr, window_mins=window_mins)
    return audio

dc_rms_stat = dc_rms_stationary(segment, sr, segment_mins=5)
dc_rms_non_stat = dc_rms_non_stationary(segment, sr, window_mins=5)

5분 세그먼트 처리: 1


In [19]:
Audio(dc_rms_stat, rate=sr)

<IPython.lib.display.Audio object>

In [20]:
Audio(dc_rms_non_stat, rate=sr)

<IPython.lib.display.Audio object>

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Glyph.*missing from font.*')

# 시각화 비교
fig, axes = plt.subplots(4,2, figsize=(15,12))

# 파형 비교
axes[0,0].plot(segment)
axes[0,0].set_title("Original Segment Waveform")
axes[0,0].set_ylabel("Amplitude")

axes[1,0].plot(stationary_audio)
axes[1,0].set_title("Stationary noise reduction Waveform")
axes[1,0].set_ylabel("Amplitude")

axes[2,0].plot(non_stationary_audio)
axes[2,0].set_title("Non-Stationary noise reduction Waveform")
axes[2,0].set_ylabel("Amplitude")

axes[3,0].plot(dc_rms_stat)
axes[3,0].set_title("DC+RMS+Stationary noise reduction Waveform")
axes[3,0].set_ylabel("Amplitude")

# 스펙트로그램 비교 
mel_spec_stat = librosa.feature.melspectrogram(y=segment, sr=sr)
mel_spec_db_stat = librosa.power_to_db(mel_spec_stat, ref=np.max)
librosa.display.specshow(mel_spec_db_stat, sr=sr, x_axis='time', y_axis='mel', ax=axes[0,1])
axes[0, 1].set_title('Original Segment Spectrogram')

# Stationary 스펙트로그램
mel_spec_stat = librosa.feature.melspectrogram(y=stationary_audio, sr=sr)
mel_spec_db_stat = librosa.power_to_db(mel_spec_stat, ref=np.max)
librosa.display.specshow(mel_spec_db_stat, sr=sr, x_axis='time', y_axis='mel', ax=axes[1, 1])
axes[1, 1].set_title('Stationary Spectrogram')

# Non-Stationary 스펙트로그램
mel_spec_stat = librosa.feature.melspectrogram(y=non_stationary_audio, sr=sr)
mel_spec_db_stat = librosa.power_to_db(mel_spec_stat, ref=np.max)
librosa.display.specshow(mel_spec_db_stat, sr=sr,
                         x_axis='time', y_axis='mel', ax=axes[2, 1])
axes[2, 1].set_title('Non-Stationary Spectrogram')

# DC+RMS+Stationary 스펙트로그램
mel_spec_dc_rms = librosa.feature.melspectrogram(y=dc_rms_stat, sr=sr)
mel_spec_db_dc_rms = librosa.power_to_db(mel_spec_dc_rms, ref=np.max)
librosa.display.specshow(mel_spec_db_dc_rms, sr=sr, x_axis='time', y_axis='mel', ax=axes[3, 1])
axes[3, 1].set_title('DC+RMS+Stationary Spectrogram')

plt.tight_layout(pad=1.0)
plt.show()

# 결과 저장
from utils.visulization import save_figure
save_figure(fig, "noise_reduction_comparison.png")

<Figure size 1500x1200 with 8 Axes>

- RMS 정규화가 노이즈를 증폭시킬 수 있습니다. 
  - data * scaling_factor 부분이 노이즈 증폭시킬수도 있습니다. 
- DC offset의 부작용이 있습니다. 
  - 일부 유용한 저주파의 성분을 제거할 수 있습니다. 
  - 코골이/호흡의 저주파 성분이 손상될 수 있습니다.

- 시각화 시, Stationary 노이즈 제거가 가장 효과적으로 보입니다.

# wiener, median filter

In [25]:
import scipy.signal

wiener_denoised = scipy.signal.wiener(segment)
print(f"타입 {type(wiener_denoised)}")
print(f"길이 {len(wiener_denoised)}")

타입 <class 'numpy.ndarray'>
길이 4800000


In [26]:
# 시각화
from utils.visulization import save_figure
fig, axes = plt.subplots(2,2, figsize=(15,8))

# Waveform 비교
axes[0, 0].plot(segment)
axes[0, 0].set_title("Original Segment Waveform")
axes[0, 0].set_ylabel("Amplitude")

axes[1, 0].plot(wiener_denoised)
axes[1, 0].set_title("Wiener Filter applied Waveform")
axes[1, 0].set_ylabel("Amplitude")

# Spectrogram 비교
mel_spec = librosa.feature.melspectrogram(y=segment, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[0,1])
axes[0,1].set_title("Original Segment Spectrogram")

mel_spec_wiener = librosa.feature.melspectrogram(y=wiener_denoised, sr=sr)
mel_spec_wiener_db = librosa.power_to_db(mel_spec_wiener, ref=np.max)
librosa.display.specshow(mel_spec_wiener_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1,1])
axes[1,1].set_title("Wiener Filter applied Spectrogram")

plt.tight_layout(pad=1.0)
plt.show()

# 결과 저장
save_figure(fig, "noise_reduction_wienerfilter.png")

<Figure size 1500x800 with 4 Axes>

In [24]:
wiener_denoised1 = scipy.signal.wiener(stationary_audio)
print(f"타입 {type(wiener_denoised1)}")
print(f"길이 {len(wiener_denoised1)}")

타입 <class 'numpy.ndarray'>
길이 4800000


In [25]:
# 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Waveform 비교
axes[0, 0].plot(stationary_audio)
axes[0, 0].set_title("Stationary-denoised Waveform")
axes[0, 0].set_ylabel("Amplitude")

axes[1, 0].plot(wiener_denoised1)
axes[1, 0].set_title("Wiener Filter applied Waveform")
axes[1, 0].set_ylabel("Amplitude")

# Spectrogram 비교
mel_spec = librosa.feature.melspectrogram(y=stationary_audio, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
librosa.display.specshow(
    mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[0, 1])
axes[0, 1].set_title("Stationary-denoised Spectrogram")

mel_spec_wiener = librosa.feature.melspectrogram(y=wiener_denoised1, sr=sr)
mel_spec_wiener_db = librosa.power_to_db(mel_spec_wiener, ref=np.max)
librosa.display.specshow(mel_spec_wiener_db, sr=sr,
                         x_axis='time', y_axis='mel', ax=axes[1, 1])
axes[1, 1].set_title("Wiener Filter applied Spectrogram")

plt.tight_layout(pad=1.0)
plt.show()

<Figure size 1500x800 with 4 Axes>

- wiener 노이즈 제거해도 큰 차이 없음
  - 오디오에는 적합하지 않습니다.
  - 노이즈가 심하면 고려해볼수 있는 옵션입니다.


| 특징 | `noisereduce` | `scipy.signal.wiener` |
|------|---------------|----------------------|
| **알고리즘** | Spectral Gating | Wiener Filter |
| **도메인** | 주파수 (STFT) | 시간 (또는 2D) |
| **용도** | 오디오 노이즈 제거 | 이미지/신호 스무딩 |
| **노이즈 추정** | 자동 또는 샘플 제공 | 국소 통계 기반 |
| **파라미터** | 많음 (세밀 조정 가능) | 간단함 (커널 크기만) |

In [ ]:
# 예제는 실행할 수 있으나, 오디오 데이터에 부적합

import pykalman
from pykalman import KalmanFilter

# 신호 추적/예측기반
kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)
# 초기상태 추정값. 
# n_dim_obs 는 관측 벡터의 차원수를 의미 
# 즉, 관측치가 몇 개의 요소로 이루어져 있는지 정의 

kf_audio = kf.filter(segment)[0]

- 코골이 같은 데이터에는 적합하지 않음음
- 칼만 필터의 작동 원리 간단히 설명:
  - 칼만 필터는 시스템 상태를 추정하는 알고리즘으로, 상태 추정값과 관측값을 반복적으로 업데이트하며 최적의 추정값을 찾습니다.

In [14]:
# median filter 추가

median_audio = scipy.signal.medfilt(stationary_audio, kernel_size=5)
print(f"타입 {type(median_audio)}")
print(f"길이 {len(median_audio)}")

타입 <class 'numpy.ndarray'>
길이 4800000


In [20]:
fig, axes = plt.subplots(2, 2, figsize=(15,10))

# Waveform 비교
axes[0, 0].plot(stationary_audio)
axes[0, 0].set_title("Stationary-denoised Waveform")
axes[0, 0].set_ylabel("Amplitude")

axes[1, 0].plot(median_audio)
axes[1, 0].set_title("Median Filter applied Waveform")


mel_spec = librosa.feature.melspectrogram(y=stationary_audio, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
librosa.display.specshow(
    mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[0, 1])
axes[0, 1].set_title("Stationary-denoised Spectrogram")

mel_spec_median = librosa.feature.melspectrogram(y=median_audio, sr=sr)
mel_spec_median_db = librosa.power_to_db(mel_spec_median, ref=np.max)
librosa.display.specshow(
    mel_spec_median_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1, 1])
axes[1, 1].set_title("Median Filter applied Spectrogram")

Text(0.5, 1.0, 'Median Filter applied Spectrogram')

<Figure size 1500x1000 with 4 Axes>

In [21]:
# 5-10ms 정도 권장
kernel_duration_ms = 10
kernel_size = int(16000 * 10 / 1000)  # = 160
kernel_size = 161  # 홀수로 조정

# 너무 크면 신호 왜곡, 너무 작으면 효과 없음

In [22]:
median_audio = scipy.signal.medfilt(stationary_audio, kernel_size=kernel_size)

In [23]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Waveform 비교
axes[0, 0].plot(stationary_audio)
axes[0, 0].set_title("Stationary-denoised Waveform")
axes[0, 0].set_ylabel("Amplitude")

axes[1, 0].plot(median_audio)
axes[1, 0].set_title("Median Filter applied Waveform")


mel_spec = librosa.feature.melspectrogram(y=stationary_audio, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
librosa.display.specshow(
    mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[0, 1])
axes[0, 1].set_title("Stationary-denoised Spectrogram")

mel_spec_median = librosa.feature.melspectrogram(y=median_audio, sr=sr)
mel_spec_median_db = librosa.power_to_db(mel_spec_median, ref=np.max)
librosa.display.specshow(
    mel_spec_median_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1, 1])
axes[1, 1].set_title("Median Filter applied Spectrogram")

Text(0.5, 1.0, 'Median Filter applied Spectrogram')

<Figure size 1500x1000 with 4 Axes>

# band pass filter

In [27]:
from scipy.signal import butter, filtfilt

def bandpass_filter(audio, lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut /nyquist
    high = highcut / nyquist
    b,a = butter(order, [low, high], btype='band')
    filtered_data = filtfilt(b, a, audio)

    return filtered_data

In [28]:
band_filtered = bandpass_filter(stationary_audio, lowcut=30, highcut=500, fs=sr)

print(f"타입 {type(band_filtered)}")
print(f"길이 {len(band_filtered)}")

타입 <class 'numpy.ndarray'>
길이 4800000


In [29]:
# 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Waveform 비교
axes[0, 0].plot(stationary_audio)
axes[0, 0].set_title("Stationary-denoised Waveform")
axes[0, 0].set_ylabel("Amplitude")

axes[1, 0].plot(band_filtered)
axes[1, 0].set_title("Bandpass Filtered Waveform")
axes[1, 0].set_ylabel("Amplitude")

# Spectrogram 비교
mel_spec = librosa.feature.melspectrogram(y=stationary_audio, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
librosa.display.specshow(
    mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[0, 1])
axes[0, 1].set_title("Stationary-denoised Spectrogram")

mel_spec_band = librosa.feature.melspectrogram(y=band_filtered, sr=sr)
mel_spec_band_db = librosa.power_to_db(mel_spec_band, ref=np.max)
librosa.display.specshow(mel_spec_band_db, sr=sr,
                         x_axis='time', y_axis='mel', ax=axes[1, 1])
axes[1, 1].set_title("Bandpass Filtered Spectrogram")

plt.tight_layout(pad=1.0)
plt.show()

<Figure size 1500x800 with 4 Axes>

In [30]:
display(Audio(band_filtered, rate=sr))

<IPython.lib.display.Audio object>

# 소음 프로파일을 적용

In [ ]:
# TEST
CHUNK = [0.1, -0.2, 0.15, -0.1, 0.05]
# 제곱으로 모든 값 양수로 바꿈
CHUNK_SQUARED = [value ** 2 for value in CHUNK]
print(f"chunk squared: {CHUNK_SQUARED}")
MEAN = sum(CHUNK_SQUARED) / 5
print(MEAN)

chunk squared: [0.010000000000000002, 0.04000000000000001, 0.0225, 0.010000000000000002, 0.0025000000000000005]
0.017000000000000005


In [ ]:
RMS= np.sqrt(MEAN)
print(RMS) 
# 실제 신호의 크기를 반영

0.130384048104053


- 오디오 신호의 평균적인 에너지/크기
- 음량(loudness)의 표준 측정 방법
- 값이 클수록 소리가 큼

In [ ]:
def find_quiet_noise_profile(audio, sr, chunk_duration=5, percentile=10):
    """
    말소리 없는(에너지가 낮은) 조용한 구간을 소음 프로파일로 사용
    """
    # 5분 청크로 나누어 에너지 계산
    chunk_size = int(chunk_duration * sr)
    max_duration = min(len(audio), int(60 * 60 * sr))  # 최대 1시간

    chunk_data = []
    # 슬라이딩 윈도우로 더 많은 샘플 수집 (50% overlap)
    hop_size = chunk_size // 2

    for i in range(0, max_duration - chunk_size, hop_size):  # 첫 1시간만 확인
        chunk = audio[i:i+chunk_size]

        rms_energy = np.sqrt(np.mean(chunk**2))
       # zero-crossing rate (말소리 판단 기준)
        zcr = np.mean(librosa.zero_crossings(chunk))

    # 가장 조용한 청크 찾기
    quietest_start = min(chunk_data, key=lambda x: x[1])[0]
    noise_sample = audio[quietest_start:quietest_start + chunk_size]

    print(
        f"가장 조용한 구간: {quietest_start/sr/60:.1f}분 ~ {(quietest_start+chunk_size)/sr/60:.1f}분")

    return noise_sample

In [ ]:
def find_quiet_noise_profile(audio, sr, chunk_duration=5, percentile=10):
    """
    말소리 없는(에너지가 낮은) 조용한 구간을 소음 프로파일로 사용
    """
    # 5분 청크로 나누어 에너지 계산
    chunk_size = int(chunk_duration * sr)
    max_duration = min(len(audio), int(60 * 60 * sr))  # 최대 1시간

    chunk_data = []
    # 슬라이딩 윈도우로 더 많은 샘플 수집 (50% overlap)
    hop_size = chunk_size // 2

    for i in range(0, max_duration - chunk_size, hop_size):  # 첫 1시간만 확인
        chunk = audio[i:i+chunk_size]

        rms_energy = np.sqrt(np.mean(chunk**2))
       # zero-crossing rate (말소리 판단 기준)
        zcr = np.mean(librosa.zero_crossings(chunk))

    # 가장 조용한 청크 찾기
    quietest_start = min(chunk_data, key=lambda x: x[1])[0]
    noise_sample = audio[quietest_start:quietest_start + chunk_size]

    print(
        f"가장 조용한 구간: {quietest_start/sr/60:.1f}분 ~ {(quietest_start+chunk_size)/sr/60:.1f}분")

    return noise_sample

In [ ]:
import noisereduce as nr


def adaptive_noise_reduction(audio_file, sr=22050, chunk_duration=5, noise_duration_minutes=20):
    """
    구간별로 소음 프로파일을 적응적으로 적용
    처음 20분을 소음 프로파일로 사용
    """
    # 오디오 로드
    y, sr = librosa.load(audio_file, sr=sr)
    print(f"전체 오디오 길이: {len(y)/sr/3600:.1f}시간")

    # 처음 20분을 소음 샘플로 추출
    noise_duration_seconds = noise_duration_minutes * 60  # 20분 = 1200초
    noise_end_sample = int(noise_duration_seconds * sr)
    noise_end_sample = min(noise_end_sample, len(y))  # 파일 길이 초과 방지

    noise_sample = y[:noise_end_sample]
    print(
        f"소음 프로파일: 처음 {noise_duration_minutes}분 ({len(noise_sample)/sr/60:.1f}분) 사용")

    # 청크 크기 계산
    chunk_size = int(chunk_duration * sr)
    processed_chunks = []

    print(f"청크별 처리 시작... (청크 크기: {chunk_duration}초)")

    for i in range(0, len(y), chunk_size):
        chunk = y[i:i + chunk_size]

        # 처음 20분의 소음 프로파일을 모든 청크에 적용
        reduced_chunk = nr.reduce_noise(
            y=chunk,
            sr=sr,
            y_noise=noise_sample,  # 처음 20분을 소음 프로파일로 사용
            stationary=False,  # 비정상 소음 처리
            prop_decrease=0.8  # 소음 감소 비율 (80% 감소)
        )

        processed_chunks.append(reduced_chunk)

        # 진행상황 출력
        progress = (i + chunk_size) / len(y) * 100
        if int(progress) % 10 == 0:
            print(f"진행률: {progress:.0f}%")

    return np.concatenate(processed_chunks), sr